# Popularity baseline

Our first recommendation method gives everyone the same five products. It ranks products by the percentage of eligible past orders containing them.

## Read the SQL
The first cell displays the actual query. `eligible_orders` selects allowed history, `product_counts` counts orders per product, and `history_size` supplies the denominator. The final step ranks products and keeps five. Product ID breaks ties consistently.

We use the 131,209 evaluation customers and only their prior orders before their validation baskets. This is a per-customer sequence cutoff, not a shared calendar-date cutoff.

In [ ]:
from pathlib import Path
import runpy

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / '03_Code/01_Data_Preparation/prepare_data.py').exists())
print((ROOT / '03_Code/02_Baselines/popularity_baseline.sql').read_text(encoding='utf-8'))

## Build the list
Run this with the project's `.venv` kernel. It opens the database read-only, verifies the cutoffs and saves a small check report. It doesn't score the hidden baskets or change the source data.

In [ ]:
baseline = runpy.run_path(str(ROOT / '03_Code/02_Baselines/run_popularity_baseline.py'))
recommendations = baseline['build_baseline']()
recommendations

## Read one row
Banana appears in 282,094 of 1,916,168 eligible history orders: **14.72%**. That is its share of these past baskets, not the probability that any particular customer buys it next. Banana and Bag of Organic Bananas are separate product IDs.

The five percentages don't have to sum to 100%: one basket can contain several listed products.

## Next
Discuss what this general list misses about individual customers. Then compare against a personal-history method. No accuracy or business impact is claimed yet.

# Score the popularity baseline

We compare the five general suggestions with each customer's hidden validation basket. The official train baskets remain reserved for final evaluation. This notebook displays the saved validation report; the last cell can reproduce it.

## 1. One customer
Customer 1 is our consistent teaching example, not a customer chosen for a high or low score.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / '03_Code/01_Data_Preparation/prepare_data.py').exists())
report = json.loads((ROOT / '05_Results/03_Checks/popularity_validation_scores.json').read_text())
example = report['example_user_1']
print(f"Customer 1: {example['matched_products']} matches in {example['actual_products']} actual products.")
pd.DataFrame(example['basket'])

## 2. Across all customers
Each customer receives five suggestions. Mean precision = average of matches / 5. Mean recall = average of matches / that customer's actual basket size. Hit rate = customers with at least one match / all evaluated customers.

Don't average the percentage of historical orders shown in the popularity list: those are inputs, not accuracy scores.

In [ ]:
pd.DataFrame([
    {'Metric': 'Mean precision at 5', 'Percent': report['mean_precision_at_5'] * 100},
    {'Metric': 'Mean recall at 5', 'Percent': report['mean_recall_at_5'] * 100},
    {'Metric': 'Hit rate at 5', 'Percent': report['hit_rate_at_5'] * 100},
]).round(2)

## 3. What this establishes
This is our validation benchmark, not proof of useful shopping suggestions. We haven't compared a personal method yet. Every later method must use these same customers and target baskets. The all-product popularity list can recommend new-to-customer products; a rebuy-only personal method has a narrower candidate set, which we'll document rather than hide.

## Optional: reproduce the scores
The next cell reads the database and rewrites only the generated validation outputs and report. No final basket is scored.

In [ ]:
import runpy
scorer = runpy.run_path(str(ROOT / '03_Code/02_Baselines/score_popularity_validation.py'))
report, customer_scores = scorer['score_validation']()
customer_scores.head()

# Personal favourites

This method recommends previously bought products. It sorts by the number of past orders containing each product, then the most recent purchase order, then product ID. **Recency only breaks ties here; this isn't a weighted-recency model.**

We use the same validation customers and hidden baskets as the general baseline. Final train baskets remain unscored.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / '03_Code/01_Data_Preparation/prepare_data.py').exists())
report = json.loads((ROOT / '05_Results/03_Checks/personal_frequency_validation.json').read_text())
pd.DataFrame(report['example_user_1']['suggestions'])[['recommendation_rank', 'product_name', 'purchase_orders', 'last_purchase_order_number']]

## One customer
Customer 1 receives Soda, Original Beef Jerky, Pistachios, Organic String Cheese and Zero Calorie Cola. All five appear in the nine-product validation basket: precision = 5/5 = 100%, recall = 5/9 = 55.56%. The general list matched none. This customer was our existing teaching example, not selected based on the score. One customer doesn't establish overall performance.

In [ ]:
pd.DataFrame([
    {'Metric': metric, 'Popularity (%)': values['popularity'] * 100,
     'Personal (%)': values['personal_frequency'] * 100,
     'Change (percentage points)': values['difference_percentage_points']}
    for metric, values in report['comparison'].items()
]).round(2)

## What to take from the comparison
The personal method scores higher on this validation sample. That doesn't establish causal shopping benefits or final-test performance. Both methods use the same 131,209 customers and baskets. Candidate strategies differ: general popularity can suggest products the customer hasn't bought; personal frequency is rebuy-only.

3,521 customers have fewer than five candidates. No popularity filler was added. Precision still divides by five, so empty slots don't earn credit. Recall divides by all products in the actual basket, including new products.

Paired uncertainty intervals are saved in the report. They rely on independent-user assumptions and don't establish that this historical sample represents all shoppers. We haven't selected a final method.

## Optional: inspect the SQL or rerun
Use the project's .venv kernel. Rerunning writes generated validation outputs and the comparison report; it doesn't change the source database.

In [ ]:
print((ROOT / '03_Code/02_Baselines/personal_frequency.sql').read_text())

In [ ]:
import runpy
method = runpy.run_path(str(ROOT / '03_Code/02_Baselines/score_personal_frequency.py'))
report, customer_scores, recommendations = method['run_comparison']()
customer_scores.head()

# Give recent purchases more weight

## 1. Choose a rule before scoring
We fixed a five-order half-life for this first experiment. A purchase in the latest eligible order gets 1 point; one five order positions older gets 0.5; ten older gets 0.25. Add the points for each product and take the five highest scores. Ties use latest purchase order, then product ID.

The age is measured in order positions, **not days**. An old purchase isn't deleted; it contributes less. This changes the ranking but still only considers previously bought products.

In [ ]:
import pandas as pd
pd.DataFrame({'Order positions older': [0, 5, 10], 'Weight': [1.0, 0.5, 0.25]})

## 2. Score the same hidden baskets
Both methods recommend up to five products from the same personal history and are evaluated on the same 131,209 customers and validation baskets. Empty slots still count against precision's denominator of five. Recall is matched / actual basket size per customer, then averaged. Hit rate is customers with any match / evaluated customers. Final train baskets remain unscored.

In [ ]:
from pathlib import Path
import json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / '03_Code/01_Data_Preparation/prepare_data.py').exists())
report = json.loads((ROOT / '05_Results/03_Checks/weighted_recency_validation.json').read_text())
pd.DataFrame([
    {'Metric': metric, 'Frequency (%)': v['personal_frequency'] * 100,
     'Recency (%)': v['weighted_recency'] * 100,
     'Difference (percentage points)': v['difference_percentage_points']}
    for metric, v in report['comparison'].items()
]).round(2)

## 3. Read the result
Mean recall rose from 23.44% to 24.54%: **+1.10 percentage points**. This is a modest validation gain, not a 1.10% increase in sales. The approximate paired sample interval is +1.06 to +1.15 points under independent-user assumptions. It doesn't establish population representativeness or causal value.

This is one chosen rule, not proof that five orders is the best setting. We haven't measured runtime trade-offs, checked subgroup regressions for this method, or selected a final model. We haven't fixed the new-product discovery limitation.

## Optional: read the SQL or reproduce
Use the project's .venv kernel. Reproduction rewrites generated validation outputs only.

In [ ]:
print((ROOT / '03_Code/02_Baselines/weighted_recency.sql').read_text())

In [ ]:
import runpy
experiment = runpy.run_path(str(ROOT / '03_Code/02_Baselines/score_weighted_recency.py'))
report, scores, recommendations = experiment['run_experiment']()
recommendations.head()

## ML preparation: SQLite feature preview

This section reads the checked feature table without importing DuckDB. The earlier lesson cells retain their original code and saved outputs; Run All still encounters the Windows DuckDB block in those historical sections. Run this preview cell independently.

Each row is one customer/product/prediction point. Training predicts the basket immediately before validation. Only earlier orders supply the five inputs. `bought_next` is the answer, not an input. Customer/product IDs and cutoff fields are identifiers for checking rows, not model features. Final official train baskets aren't loaded.

Build once with `03_Code/03_Model_Comparison/build_ml_features.py` using the project interpreter. The script checks the rows and compares the two existing personal methods with their saved validation metrics. No ML model is fitted.


In [ ]:
# Locate the project from the notebook's working directory.
from pathlib import Path
import json
import sqlite3
import pandas as pd
project_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / '02_Data/02_Prepared/evaluation_plan.csv').exists())
feature_db = project_root / '02_Data/02_Prepared/instacart_ml.sqlite'
feature_report = json.loads((project_root / '05_Results/03_Checks/ml_feature_preparation.json').read_text(encoding='utf-8'))
assert feature_report['status'] == 'passed'

# Open read-only and show a few real feature rows, without rebuilding or fitting anything.
with sqlite3.connect(feature_db.as_uri() + '?mode=ro', uri=True) as connection:
    assert connection.execute("SELECT value FROM build_metadata WHERE key='features_checked'").fetchone() == ('passed',)
    preview = pd.read_sql_query("""SELECT stage,user_id,product_id,history_orders,
        purchase_orders,purchase_share,orders_since_last_purchase,recency_score,bought_next
        FROM ml_features WHERE user_id=1
        ORDER BY stage,purchase_orders DESC,last_purchase_order DESC,product_id LIMIT 12""", connection)
display(preview)


## First logistic-regression challenger: saved validation results

Scaling rewrites inputs using training averages and spreads. Regularisation restrains large model weights. The pipeline learns those adjustments and logistic-regression weights from the training examples only. The same scaler is reused for validation.

The first fixed attempt uses the five prepared inputs, L2 C=1.0, all training candidate rows and no class balancing. It ranks up to five familiar products per customer. This cell reads saved results; it doesn't retrain. Old DuckDB cells aren't needed. To reproduce this attempt separately, use `03_Code/03_Model_Comparison/train_logistic_validation.py` with the project interpreter.

These are exploratory validation scores, not final scores or customer-benefit measurements. Correlated input weights aren't causal explanations. ID, cutoff and answer columns aren't model inputs.


In [ ]:
# Read the checked model comparison without rerunning training.
from pathlib import Path
import json
import pandas as pd
project_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / '02_Data/02_Prepared/evaluation_plan.csv').exists())
model_report = json.loads((project_root / '05_Results/03_Checks/logistic_validation.json').read_text(encoding='utf-8'))
assert model_report['status'] == 'passed'
comparison = pd.DataFrame(model_report['comparison']).T
comparison[['logistic_regression', 'weighted_recency']] *= 100
comparison = comparison.rename(columns={'logistic_regression': 'Logistic (%)', 'weighted_recency': 'Weighted recency (%)', 'difference_percentage_points': 'Difference (percentage points)'})
display(comparison[['Logistic (%)', 'Weighted recency (%)', 'Difference (percentage points)']])


## Boosted-tree challenger: saved validation results

Decision trees learn sequences of questions. Boosting adds small trees that adjust the earlier predictions. This fixed attempt uses the same five inputs and identical training/validation baskets as logistic regression. It uses raw inputs, 100 small-tree rounds, no automatic random-row holdout and no final basket answers.

The working target of +0.5 percentage points recall versus weighted recency was stated before seeing this attempt's result; precision, hit rate, segments and runtime still matter. This is an experiment target, not a proven customer-benefit or deployment threshold.

The cell reads saved evidence without training again. To reproduce the attempt separately, use `03_Code/03_Model_Comparison/train_boosted_validation.py`. It reuses the existing validation runner and checks, preserving the earlier logistic files.


In [ ]:
# Read the fixed tree experiment without fitting another model.
from pathlib import Path
import json
import pandas as pd
project_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / '02_Data/02_Prepared/evaluation_plan.csv').exists())
tree_report = json.loads((project_root / '05_Results/03_Checks/boosted_trees_validation.json').read_text(encoding='utf-8'))
assert tree_report['status'] == 'passed'
comparison_rows = []
for metric, result in tree_report['comparison'].items():
    comparison_rows.append({'Measure': metric, 'Weighted recency (%)': 100 * result['weighted_recency'],
        'Logistic (%)': 100 * tree_report['comparison_with_logistic'][metric]['logistic_regression'],
        'Boosted trees (%)': 100 * result['boosted_trees'],
        'Tree gain vs recency (pp)': result['difference_percentage_points']})
display(pd.DataFrame(comparison_rows))


## Version 2: purchase timing — saved development result
This reads the saved report only. The first fixed rule failed its development gate; the 15,000-person audit remains unscored. V1 stays unchanged. Different cohort/targets mean these scores must not be compared directly with V1 finals.


In [ ]:
# Read the checked comparison without rebuilding inputs or evaluating any baskets.
from pathlib import Path
import json
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / '03_Code/03_Model_Comparison/evaluate_purchase_timing.py').exists())
result = json.loads((ROOT / '05_Results/03_Checks/v2_timing_development.json').read_text(encoding='utf-8'))
assert result['status'] == 'passed' and result['stage'] == 'development'
display(pd.DataFrame(result['overall']).T.mul(100).round(3))
print('Recall difference (percentage points):', round(result['timing_minus_recency_pp']['recall_at_5'], 4))
print('Passed fixed development gate:', result['passes_prospective_gate'])
print('Next action:', result['next_action'])


## Version 3: discovery — saved development results
Compare unfamiliar popularity with related products. Development answers were previously exposed; the reserved audit remains unused. Recall here uses only baskets with novel products.

In [ ]:
# Read the saved discovery comparison; this doesn't train or rescore anything.
import json
from pathlib import Path
import pandas as pd
project = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '05_Results/03_Checks/discovery_development.json').exists())
r = json.loads((project / '05_Results/03_Checks/discovery_development.json').read_text())
metrics = ['hit_at_3', 'precision_at_3', 'recall_at_3_among_novel_baskets']
print(pd.DataFrame(r['overall']).T[metrics].mul(100).round(3))
print('All shoppers:', r['customers'], '| Novel-basket recall denominator:', r['customers_with_novel_products'])
print('Exploratory development gate:', r['passes_exploratory_gate'], '| Audit used:', r['audit_used'])


## Version 3: reserved audit — completed
Same fixed methods; no refit. Audit answers are now exposed. Results support considering a discovery prototype, not a causal customer-benefit claim.

In [ ]:
# Read the frozen audit result without rebuilding or rescoring.
import json
from pathlib import Path
import pandas as pd
project = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '05_Results/03_Checks/discovery_audit.json').exists())
audit = json.loads((project / '05_Results/03_Checks/discovery_audit.json').read_text())
columns = ['hit_at_3', 'precision_at_3', 'recall_at_3_among_novel_baskets']
print(pd.DataFrame(audit['overall']).T[columns].mul(100).round(3).to_string())
print('All shoppers:', audit['customers'], '| Novel-recall denominator:', audit['customers_with_novel_products'])
print('Gate passed:', audit['passes_fixed_audit_gate'], '| Audit now exposed:', audit['audit_now_exposed'])
